In [1]:
import os
import cv2
import torch
import numpy as np
import pandas as pd

import torch.nn as nn
import torch.optim as optim

from PIL import Image
from tqdm import tqdm

from torchvision import transforms
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [2]:
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(DEVICE)

cuda


In [3]:
PAIR_ROOT = r"D:\ARTEN_Project\TrainingPairs"

INPUT_DIR = os.path.join(
    PAIR_ROOT,
    "input"
)

TARGET_DIR = os.path.join(
    PAIR_ROOT,
    "target"
)

In [4]:
class ARTENDataset(Dataset):

    def __init__(
        self,
        input_dir,
        target_dir
    ):

        self.input_dir = input_dir
        self.target_dir = target_dir

        self.files = sorted(
            os.listdir(input_dir)
        )

        self.transform = transforms.Compose([

            transforms.ToPILImage(),

            transforms.ToTensor()

        ])

    def __len__(self):

        return len(self.files)

    def __getitem__(self, idx):

        name = self.files[idx]

        inp = cv2.imread(
            os.path.join(
                self.input_dir,
                name
            )
        )

        tgt = cv2.imread(
            os.path.join(
                self.target_dir,
                name
            )
        )

        inp = cv2.cvtColor(
            inp,
            cv2.COLOR_BGR2RGB
        )

        tgt = cv2.cvtColor(
            tgt,
            cv2.COLOR_BGR2RGB
        )

        inp = self.transform(inp)

        tgt = self.transform(tgt)

        return inp, tgt

In [5]:
dataset = ARTENDataset(
    INPUT_DIR,
    TARGET_DIR
)

print(
    "Dataset Size:",
    len(dataset)
)

Dataset Size: 21875


In [6]:
train_loader = DataLoader(

    dataset,

    batch_size=16,

    shuffle=True,

    num_workers=0
)

In [7]:
class SpatialAttention(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv = nn.Conv2d(

            2,
            1,
            kernel_size=7,
            padding=3

        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = torch.mean(
            x,
            dim=1,
            keepdim=True
        )

        max_out, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        x_cat = torch.cat(
            [avg_out, max_out],
            dim=1
        )

        attention = self.sigmoid(
            self.conv(x_cat)
        )

        return attention

In [8]:
class ResidualAttentionBlock(nn.Module):

    def __init__(
        self,
        channels
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(

            channels,
            channels,
            3,
            padding=1

        )

        self.relu = nn.ReLU()

        self.conv2 = nn.Conv2d(

            channels,
            channels,
            3,
            padding=1

        )

        self.attention = SpatialAttention()

    def forward(self, x):

        residual = x

        out = self.conv1(x)

        out = self.relu(out)

        out = self.conv2(out)

        A = self.attention(out)

        out = out * A

        return residual + out

In [9]:
block = ResidualAttentionBlock(
    64
)

dummy = torch.randn(
    1,
    64,
    256,
    256
)

output = block(dummy)

print(output.shape)

torch.Size([1, 64, 256, 256])


In [10]:
attention = SpatialAttention()

dummy = torch.randn(
    1,
    64,
    256,
    256
)

A = attention(dummy)

print(A.shape)

torch.Size([1, 1, 256, 256])


In [11]:
class EncoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=1
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(),

            ResidualAttentionBlock(
                out_channels
            )

        )

        self.pool = nn.MaxPool2d(2)

    def forward(self, x):

        features = self.block(x)

        pooled = self.pool(features)

        return features, pooled

In [12]:
class DecoderBlock(nn.Module):

    def __init__(self, in_channels, skip_channels, out_channels):

        super().__init__()

        self.up = nn.ConvTranspose2d(

            in_channels,
            out_channels,
            kernel_size=2,
            stride=2

        )

        self.conv = nn.Sequential(

            nn.Conv2d(

                out_channels + skip_channels,
                out_channels,
                3,
                padding=1

            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(),

            ResidualAttentionBlock(
                out_channels
            )

        )

    def forward(self, x, skip):

        x = self.up(x)

        x = torch.cat(
            [x, skip],
            dim=1
        )

        x = self.conv(x)

        return x

In [13]:
class MultiScaleHead(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.head = nn.Conv2d(
            channels,
            3,
            1
        )

    def forward(self, x):

        return torch.sigmoid(
            self.head(x)
        )

In [14]:
def clarity_gate(

    clarity_score,

    tau,

    kappa=0.05,

    bias=0

):

    gate = 1 / (

        1 +

        np.exp(

            -(kappa *
              (tau - clarity_score)
              + bias)

        )

    )

    return gate

In [17]:
with open("tau_threshold.txt", "r") as f:

    tau = float(f.read())

print("Loaded tau =", tau)

Loaded tau = 41.3657885634315


In [18]:
print(
    clarity_gate(
        50,
        tau
    )
)

0.39371793687216783


In [19]:
class ARTEN(nn.Module):

    def __init__(self):

        super().__init__()

        self.enc1 = EncoderBlock(
            3,
            64
        )

        self.enc2 = EncoderBlock(
            64,
            128
        )

        self.enc3 = EncoderBlock(
            128,
            256
        )

        self.bottleneck = nn.Sequential(

            nn.Conv2d(
                256,
                512,
                3,
                padding=1
            ),

            nn.ReLU(),

            ResidualAttentionBlock(
                512
            )

        )

        self.dec3 = DecoderBlock(
            512,
            256,
            256
        )

        self.dec2 = DecoderBlock(
            256,
            128,
            128
        )

        self.dec1 = DecoderBlock(
            128,
            64,
            64
        )

        self.final = nn.Conv2d(
            64,
            3,
            1
        )

        self.ms1 = MultiScaleHead(256)
        self.ms2 = MultiScaleHead(128)
        self.ms3 = MultiScaleHead(64)

    def forward(self, x):

        s1, p1 = self.enc1(x)

        s2, p2 = self.enc2(p1)

        s3, p3 = self.enc3(p2)

        bottleneck = self.bottleneck(p3)

        d3 = self.dec3(
            bottleneck,
            s3
        )

        d2 = self.dec2(
            d3,
            s2
        )

        d1 = self.dec1(
            d2,
            s1
        )

        out = torch.sigmoid(
            self.final(d1)
        )

        ms_out1 = self.ms1(d3)

        ms_out2 = self.ms2(d2)

        ms_out3 = self.ms3(d1)

        return (
            out,
            ms_out1,
            ms_out2,
            ms_out3
        )

In [20]:
model = ARTEN().to(
    DEVICE
)

print(model)

ARTEN(
  (enc1): EncoderBlock(
    (block): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): ResidualAttentionBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (relu): ReLU()
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (attention): SpatialAttention(
          (conv): Conv2d(2, 1, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
          (sigmoid): Sigmoid()
        )
      )
    )
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (enc2): EncoderBlock(
    (block): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): ResidualAttentionBlo

In [21]:
dummy = torch.randn(

    1,
    3,
    256,
    256

).to(DEVICE)

outputs = model(dummy)

for i, out in enumerate(outputs):

    print(
        f"Output {i}:",
        out.shape
    )

Output 0: torch.Size([1, 3, 256, 256])
Output 1: torch.Size([1, 3, 64, 64])
Output 2: torch.Size([1, 3, 128, 128])
Output 3: torch.Size([1, 3, 256, 256])


In [22]:
def attention_refinement(

    restored,

    attention,

    original

):

    refined = (

        attention * restored +

        (1 - attention) * original

    )

    return refined

In [23]:
restored = torch.rand(
    1,3,256,256
)

attention = torch.rand(
    1,1,256,256
)

original = torch.rand(
    1,3,256,256
)

refined = attention_refinement(

    restored,

    attention,

    original

)

print(refined.shape)

torch.Size([1, 3, 256, 256])


In [24]:
print(
    "Training Pairs:",
    len(dataset)
)

Training Pairs: 21875


In [25]:
dummy = torch.randn(
    1,
    3,
    256,
    256
).to(DEVICE)

outputs = model(dummy)

for i, out in enumerate(outputs):

    print(
        f"Output {i}:",
        out.shape
    )

Output 0: torch.Size([1, 3, 256, 256])
Output 1: torch.Size([1, 3, 64, 64])
Output 2: torch.Size([1, 3, 128, 128])
Output 3: torch.Size([1, 3, 256, 256])


In [26]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

CUDA Available: True
GPU: NVIDIA RTX 4500 Ada Generation


In [27]:
!pip install pytorch-msssim

In [28]:
from pytorch_msssim import ms_ssim

from torchvision.models import vgg16
from torchvision.models import VGG16_Weights

In [29]:
class PerceptualLoss(nn.Module):

    def __init__(self):

        super().__init__()

        vgg = vgg16(
            weights=VGG16_Weights.DEFAULT
        )

        self.features = nn.Sequential(
            *list(vgg.features)[:16]
        )

        for param in self.features.parameters():

            param.requires_grad = False

    def forward(self, pred, target):

        pred_feat = self.features(pred)

        target_feat = self.features(target)

        return nn.functional.l1_loss(
            pred_feat,
            target_feat
        )

In [30]:
perceptual_loss = PerceptualLoss().to(
    DEVICE
)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\admin/.cache\torch\hub\checkpoints\vgg16-397923af.pth
100%|████████████████████████████████████████████████████████████████████████████████| 528M/528M [00:04<00:00, 116MB/s]


In [31]:
def edge_loss(pred, target):

    sobel_x = torch.tensor(

        [[-1,0,1],
         [-2,0,2],
         [-1,0,1]],

        dtype=torch.float32

    ).to(DEVICE)

    sobel_x = sobel_x.view(
        1,1,3,3
    )

    pred_gray = torch.mean(
        pred,
        dim=1,
        keepdim=True
    )

    target_gray = torch.mean(
        target,
        dim=1,
        keepdim=True
    )

    pred_edge = nn.functional.conv2d(

        pred_gray,

        sobel_x,

        padding=1

    )

    target_edge = nn.functional.conv2d(

        target_gray,

        sobel_x,

        padding=1

    )

    return nn.functional.l1_loss(

        pred_edge,

        target_edge

    )

In [32]:
def ms_ssim_loss(pred, target):

    return 1 - ms_ssim(

        pred,

        target,

        data_range=1,

        size_average=True

    )

In [33]:
def multiscale_loss(

    ms1,
    ms2,
    ms3,
    target

):

    target_64 = nn.functional.interpolate(

        target,

        size=(64,64)

    )

    target_128 = nn.functional.interpolate(

        target,

        size=(128,128)

    )

    l1 = nn.functional.l1_loss(
        ms1,
        target_64
    )

    l2 = nn.functional.l1_loss(
        ms2,
        target_128
    )

    l3 = nn.functional.l1_loss(
        ms3,
        target
    )

    return l1 + l2 + l3

In [34]:
def hybrid_loss(

    pred,
    ms1,
    ms2,
    ms3,
    target

):

    l1 = nn.functional.l1_loss(
        pred,
        target
    )

    msssim = ms_ssim_loss(
        pred,
        target
    )

    perceptual = perceptual_loss(
        pred,
        target
    )

    edge = edge_loss(
        pred,
        target
    )

    multi = multiscale_loss(

        ms1,
        ms2,
        ms3,
        target

    )

    total_loss = (

        1.0 * l1 +

        0.5 * msssim +

        0.1 * perceptual +

        0.1 * edge +

        0.2 * multi

    )

    return total_loss

In [35]:
optimizer = optim.Adam(

    model.parameters(),

    lr=1e-4

)

In [36]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode='min',

    factor=0.5,

    patience=5

)

In [37]:
EPOCHS = 100

best_loss = float("inf")

In [38]:
inputs, targets = next(
    iter(train_loader)
)

inputs = inputs.to(DEVICE)

targets = targets.to(DEVICE)

outputs = model(inputs)

pred, ms1, ms2, ms3 = outputs

loss = hybrid_loss(

    pred,

    ms1,
    ms2,
    ms3,

    targets

)

print(loss.item())

0.747509241104126


In [39]:
train_losses = []

In [40]:
for epoch in range(EPOCHS):

    model.train()

    epoch_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for inputs, targets in progress_bar:

        inputs = inputs.to(DEVICE)

        targets = targets.to(DEVICE)

        optimizer.zero_grad()

        pred, ms1, ms2, ms3 = model(inputs)

        loss = hybrid_loss(

            pred,

            ms1,
            ms2,
            ms3,

            targets
        )

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

        progress_bar.set_postfix(

            Loss=loss.item()

        )

    avg_loss = epoch_loss / len(train_loader)

    train_losses.append(avg_loss)

    scheduler.step(avg_loss)

    print(

        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Average Loss: {avg_loss:.6f}"

    )

Epoch 1/100: 100%|█████████████████████████████████████████████████████| 1368/1368 [12:00<00:00,  1.90it/s, Loss=0.223]


Epoch [1/100] Average Loss: 0.222275


Epoch 2/100: 100%|█████████████████████████████████████████████████████| 1368/1368 [11:56<00:00,  1.91it/s, Loss=0.234]


Epoch [2/100] Average Loss: 0.190610


Epoch 3/100: 100%|█████████████████████████████████████████████████████| 1368/1368 [11:50<00:00,  1.93it/s, Loss=0.277]


...

Epoch [98/100] Average Loss: 0.099051


Epoch 99/100: 100%|███████████████████████████████████████████████████| 1368/1368 [11:53<00:00,  1.92it/s, Loss=0.0678]


Epoch [99/100] Average Loss: 0.099133


Epoch 100/100: 100%|███████████████████████████████████████████████████| 1368/1368 [11:55<00:00,  1.91it/s, Loss=0.096]

Epoch [100/100] Average Loss: 0.098729


In [41]:
MODEL_PATH = "ARTEN_Model.pth"

torch.save(
    model.state_dict(),
    MODEL_PATH
)

print("Model Saved")

Model Saved


In [42]:
import os

print(os.path.exists("ARTEN_Model.pth"))

True


In [43]:
MODEL_PATH = "ARTEN_Model.pth"

torch.save(
    model.state_dict(),
    MODEL_PATH
)

print("Model Saved")

Model Saved


In [44]:
import os

print(os.getcwd())

print(
    os.path.abspath(
        "ARTEN_Model.pth"
    )
)

print(
    os.path.exists(
        "ARTEN_Model.pth"
    )
)

C:\Users\admin
C:\Users\admin\ARTEN_Model.pth
True


In [45]:
import os

print("Current Directory:")
print(os.getcwd())

print("\nFiles in current directory:")

for file in os.listdir():
    print(file)

Current Directory:
C:\Users\admin

Files in current directory:
.anaconda
.cache
.conda
.continuum
.copilot
.ipynb_checkpoints
.ipython
.jupyter
.keras
.local
.matplotlib
.virtual_documents
.vscode
anaconda3
anaconda_projects
AppData
Application Data
ARTEN Architecture.ipynb
ARTEN Dataset Enhancement.ipynb
ARTEN_Dataset Analysis.ipynb
ARTEN_Model.pth
clarity_scores.csv
Contacts
Cookies
data
Desktop
Documents
Downloads
Favorites
hf_cache
Links
Local Settings
Music
My Documents
NetHood
NTUSER.DAT
ntuser.dat.LOG1
ntuser.dat.LOG2
NTUSER.DAT{444df0e1-cd28-11f0-bcd9-adcf6952415d}.TxR.blf
NTUSER.DAT{444df0e2-cd28-11f0-bcd9-adcf6952415d}.TM.blf
NTUSER.DAT{444df0e2-cd28-11f0-bcd9-adcf6952415d}.TMContainer00000000000000000001.regtrans-ms
NTUSER.DAT{444df0e2-cd28-11f0-bcd9-adcf6952415d}.TMContainer00000000000000000002.regtrans-ms
ntuser.ini
OneDrive
Pair Generation New.ipynb
Pair Generation.ipynb
Pictures
PrintHood
Recent
runs
Saved Games
Searches
SendTo
Start Menu
tau_threshold.txt
Templates
trai